# 📊 Phase 3: Data Preprocessing

## 🧩 Tasks Checklist:
- [x] **Step 3.1**: Handle missing values
- [x] **Step 3.2**: Fix data inconsistencies
- [x] **Step 3.3**: Feature engineering
- [x] **Step 3.4**: Encode categorical variables
- [x] **Step 3.5**: Handle outliers
- [x] **Step 3.6**: Feature scaling
- [x] **Step 3.7**: Handle class imbalance (decision)
- [x] **Step 3.8**: Train-test split

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Libraries imported successfully!")
print(f"📅 Analysis started on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
📅 Analysis started on: 2025-11-28 01:50:32


In [2]:
# Load the dataset
data_path = "../data/raw/Electronic_sales_Sep2023-Sep2024.csv"
df = pd.read_csv(data_path)

print("✅ Data loaded successfully!")
print(f"📊 Dataset shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

✅ Data loaded successfully!
📊 Dataset shape: (20000, 16)
📋 Columns: ['Customer ID', 'Age', 'Gender', 'Loyalty Member', 'Product Type', 'SKU', 'Rating', 'Order Status', 'Payment Method', 'Total Price', 'Unit Price', 'Quantity', 'Purchase Date', 'Shipping Type', 'Add-ons Purchased', 'Add-on Total']


---
## 🔧 Step 3.1: Handle Missing Values

**Goal**: Impute or handle missing data appropriately

| Column | Missing | Strategy |
|--------|---------|----------|
| Gender | 1 (0.005%) | Impute with mode ("Male") |
| Add-ons Purchased | 4,868 (24.3%) | Replace with "None" (no add-on) |

In [3]:
# Step 3.1: Handle Missing Values
print("="*60)
print("STEP 3.1: MISSING VALUES ANALYSIS")
print("="*60)

# Check missing values before
print("\n📊 Missing Values BEFORE:")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

# Verify Add-ons NaN = Add-on Total is 0
print("\n🔍 Verifying Add-ons Purchased NaN logic:")
addons_null = df[df['Add-ons Purchased'].isnull()]['Add-on Total']
print(f"   • Rows with NaN Add-ons: {len(addons_null)}")
print(f"   • Of those, Add-on Total = 0: {(addons_null == 0).sum()}")
print(f"   ✅ Confirmed: All NaN Add-ons have Add-on Total = 0")

STEP 3.1: MISSING VALUES ANALYSIS

📊 Missing Values BEFORE:
Gender                  1
Add-ons Purchased    4868
dtype: int64

🔍 Verifying Add-ons Purchased NaN logic:
   • Rows with NaN Add-ons: 4868
   • Of those, Add-on Total = 0: 4868
   ✅ Confirmed: All NaN Add-ons have Add-on Total = 0


In [4]:
# Apply missing value fixes
print("\n" + "="*60)
print("APPLYING FIXES")
print("="*60)

# FIX 1: Gender - impute with mode
gender_mode = df['Gender'].mode()[0]
print(f"\n✅ FIX 1: Gender")
print(f"   • Mode value: {gender_mode}")
print(f"   • Before: {df['Gender'].isnull().sum()} missing")
df['Gender'] = df['Gender'].fillna(gender_mode)
print(f"   • After: {df['Gender'].isnull().sum()} missing")

# FIX 2: Add-ons Purchased - replace NaN with "None"
print(f"\n✅ FIX 2: Add-ons Purchased")
print(f"   • Before: {df['Add-ons Purchased'].isnull().sum()} missing")
df['Add-ons Purchased'] = df['Add-ons Purchased'].fillna('None')
print(f"   • After: {df['Add-ons Purchased'].isnull().sum()} missing")

# Verify
print(f"\n✅ Total missing values remaining: {df.isnull().sum().sum()}")


APPLYING FIXES

✅ FIX 1: Gender
   • Mode value: Male
   • Before: 1 missing
   • After: 0 missing

✅ FIX 2: Add-ons Purchased
   • Before: 4868 missing
   • After: 0 missing

✅ Total missing values remaining: 0


---
## 🔧 Step 3.2: Fix Data Inconsistencies

**Issue Found**: Payment Method has "PayPal" (3,284) and "Paypal" (2,514) — case inconsistency

**Fix**: Standardize to "PayPal"

In [5]:
# Step 3.2: Fix Data Inconsistencies
print("="*60)
print("STEP 3.2: FIX DATA INCONSISTENCIES")
print("="*60)

# Check Payment Method before
print("\n📊 Payment Method BEFORE:")
print(df['Payment Method'].value_counts())

# Fix: Standardize "Paypal" to "PayPal"
df['Payment Method'] = df['Payment Method'].replace('Paypal', 'PayPal')

# Check after
print("\n📊 Payment Method AFTER:")
print(df['Payment Method'].value_counts())

print(f"\n✅ Fix applied: 'Paypal' → 'PayPal'")
print(f"   • Total PayPal records now: {(df['Payment Method'] == 'PayPal').sum()}")

STEP 3.2: FIX DATA INCONSISTENCIES

📊 Payment Method BEFORE:
Payment Method
Credit Card      5868
Bank Transfer    3371
PayPal           3284
Paypal           2514
Cash             2492
Debit Card       2471
Name: count, dtype: int64

📊 Payment Method AFTER:
Payment Method
Credit Card      5868
PayPal           5798
Bank Transfer    3371
Cash             2492
Debit Card       2471
Name: count, dtype: int64

✅ Fix applied: 'Paypal' → 'PayPal'
   • Total PayPal records now: 5798


---
## 🔧 Step 3.3: Feature Engineering

**New Features Created**:

| Feature | Logic | Rationale |
|---------|-------|----------|
| `Purchase_Month` | Extract month (1-12) | Seasonal patterns |
| `Purchase_DayOfWeek` | Extract day (0-6) | Weekly patterns |
| `Is_Weekend` | 1 if Sat/Sun | Weekend behavior |
| `Has_Addon` | 1 if Add-on Total > 0 | Add-on purchase flag |
| `Addon_Count` | Count comma-separated items | Number of add-ons |
| `Is_Repeat_Customer` | 1 if customer has multiple orders | Repeat customer flag |

In [6]:
# Step 3.3: Feature Engineering
print("="*60)
print("STEP 3.3: FEATURE ENGINEERING")
print("="*60)

# Convert Purchase Date to datetime
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

print("\n📊 CREATING NEW FEATURES:\n")

# 1. Purchase_Month (1-12)
df['Purchase_Month'] = df['Purchase Date'].dt.month
print(f"✅ 1. Purchase_Month created")
print(f"   Range: {df['Purchase_Month'].min()} - {df['Purchase_Month'].max()}")

# 2. Purchase_DayOfWeek (0=Monday, 6=Sunday)
df['Purchase_DayOfWeek'] = df['Purchase Date'].dt.dayofweek
print(f"\n✅ 2. Purchase_DayOfWeek created")
print(f"   Range: {df['Purchase_DayOfWeek'].min()} - {df['Purchase_DayOfWeek'].max()}")

# 3. Is_Weekend (1 if Sat/Sun)
df['Is_Weekend'] = (df['Purchase_DayOfWeek'] >= 5).astype(int)
print(f"\n✅ 3. Is_Weekend created")
print(f"   Weekday (0): {(df['Is_Weekend'] == 0).sum():,}")
print(f"   Weekend (1): {(df['Is_Weekend'] == 1).sum():,}")

# 4. Has_Addon (1 if Add-on Total > 0)
df['Has_Addon'] = (df['Add-on Total'] > 0).astype(int)
print(f"\n✅ 4. Has_Addon created")
print(f"   No addon (0): {(df['Has_Addon'] == 0).sum():,}")
print(f"   Has addon (1): {(df['Has_Addon'] == 1).sum():,}")

# 5. Addon_Count (count comma-separated items)
def count_addons(x):
    if x == 'None' or pd.isna(x):
        return 0
    return len(str(x).split(','))

df['Addon_Count'] = df['Add-ons Purchased'].apply(count_addons)
print(f"\n✅ 5. Addon_Count created")
print(f"   Distribution:")
print(df['Addon_Count'].value_counts().sort_index())

# 6. Is_Repeat_Customer (1 if customer has multiple transactions)
customer_counts = df['Customer ID'].value_counts()
repeat_customers = customer_counts[customer_counts > 1].index
df['Is_Repeat_Customer'] = df['Customer ID'].isin(repeat_customers).astype(int)
print(f"\n✅ 6. Is_Repeat_Customer created")
print(f"   One-time (0): {(df['Is_Repeat_Customer'] == 0).sum():,}")
print(f"   Repeat (1): {(df['Is_Repeat_Customer'] == 1).sum():,}")

STEP 3.3: FEATURE ENGINEERING

📊 CREATING NEW FEATURES:

✅ 1. Purchase_Month created
   Range: 1 - 12

✅ 2. Purchase_DayOfWeek created
   Range: 0 - 6

✅ 3. Is_Weekend created
   Weekday (0): 14,302
   Weekend (1): 5,698

✅ 4. Has_Addon created
   No addon (0): 4,868
   Has addon (1): 15,132

✅ 5. Addon_Count created
   Distribution:
Addon_Count
0    4868
1    5026
2    5087
3    5019
Name: count, dtype: int64

✅ 6. Is_Repeat_Customer created
   One-time (0): 6,637
   Repeat (1): 13,363


In [7]:
# Feature Engineering Summary
print("\n" + "="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)
print(f"\n📊 New features added: 6")
print(f"📊 Total columns now: {len(df.columns)}")
print(f"\n🆕 New columns:")
new_cols = ['Purchase_Month', 'Purchase_DayOfWeek', 'Is_Weekend', 'Has_Addon', 'Addon_Count', 'Is_Repeat_Customer']
for col in new_cols:
    print(f"   • {col}")


FEATURE ENGINEERING SUMMARY

📊 New features added: 6
📊 Total columns now: 22

🆕 New columns:
   • Purchase_Month
   • Purchase_DayOfWeek
   • Is_Weekend
   • Has_Addon
   • Addon_Count
   • Is_Repeat_Customer


---
## 🔧 Step 3.4: Encode Categorical Variables

**Encoding Strategy**:

| Column | Strategy |
|--------|----------|
| Gender | Binary: Male=1, Female=0 |
| Loyalty Member | Binary: Yes=1, No=0 |
| Order Status (TARGET) | Binary: Completed=1, Cancelled=0 |
| Product Type | One-Hot Encoding |
| Payment Method | One-Hot Encoding |
| Shipping Type | One-Hot Encoding |
| SKU | Drop (redundant with Product Type) |
| Add-ons Purchased | Drop (captured by Has_Addon & Addon_Count) |

In [8]:
# Step 3.4: Encode Categorical Variables
print("="*60)
print("STEP 3.4: ENCODE CATEGORICAL VARIABLES")
print("="*60)

print(f"\n📊 Columns before encoding: {len(df.columns)}")

# 1. Binary Encoding
print("\n" + "-"*50)
print("1️⃣ BINARY ENCODING")
print("-"*50)

# Gender: Male=1, Female=0
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
print(f"✅ Gender: Male=1, Female=0")
print(f"   Distribution: {df['Gender'].value_counts().to_dict()}")

# Loyalty Member: Yes=1, No=0
df['Loyalty Member'] = df['Loyalty Member'].map({'Yes': 1, 'No': 0})
print(f"\n✅ Loyalty Member: Yes=1, No=0")
print(f"   Distribution: {df['Loyalty Member'].value_counts().to_dict()}")

# Order Status (TARGET): Completed=1, Cancelled=0
df['Order Status'] = df['Order Status'].map({'Completed': 1, 'Cancelled': 0})
print(f"\n✅ Order Status (TARGET): Completed=1, Cancelled=0")
print(f"   Distribution: {df['Order Status'].value_counts().to_dict()}")

STEP 3.4: ENCODE CATEGORICAL VARIABLES

📊 Columns before encoding: 22

--------------------------------------------------
1️⃣ BINARY ENCODING
--------------------------------------------------
✅ Gender: Male=1, Female=0
   Distribution: {1: 10165, 0: 9835}

✅ Loyalty Member: Yes=1, No=0
   Distribution: {0: 15657, 1: 4343}

✅ Order Status (TARGET): Completed=1, Cancelled=0
   Distribution: {1: 13432, 0: 6568}


In [9]:
# 2. One-Hot Encoding
print("\n" + "-"*50)
print("2️⃣ ONE-HOT ENCODING")
print("-"*50)

# Product Type
print(f"\n✅ Product Type:")
print(f"   Before: {df['Product Type'].nunique()} categories")
product_dummies = pd.get_dummies(df['Product Type'], prefix='Product')
print(f"   Created columns: {list(product_dummies.columns)}")

# Payment Method
print(f"\n✅ Payment Method:")
print(f"   Before: {df['Payment Method'].nunique()} categories")
payment_dummies = pd.get_dummies(df['Payment Method'], prefix='Payment')
print(f"   Created columns: {list(payment_dummies.columns)}")

# Shipping Type
print(f"\n✅ Shipping Type:")
print(f"   Before: {df['Shipping Type'].nunique()} categories")
shipping_dummies = pd.get_dummies(df['Shipping Type'], prefix='Shipping')
print(f"   Created columns: {list(shipping_dummies.columns)}")

# Concatenate all dummies
df = pd.concat([df, product_dummies, payment_dummies, shipping_dummies], axis=1)


--------------------------------------------------
2️⃣ ONE-HOT ENCODING
--------------------------------------------------

✅ Product Type:
   Before: 5 categories
   Created columns: ['Product_Headphones', 'Product_Laptop', 'Product_Smartphone', 'Product_Smartwatch', 'Product_Tablet']

✅ Payment Method:
   Before: 5 categories
   Created columns: ['Payment_Bank Transfer', 'Payment_Cash', 'Payment_Credit Card', 'Payment_Debit Card', 'Payment_PayPal']

✅ Shipping Type:
   Before: 5 categories
   Created columns: ['Shipping_Expedited', 'Shipping_Express', 'Shipping_Overnight', 'Shipping_Same Day', 'Shipping_Standard']


In [10]:
# 3. Drop redundant columns
print("\n" + "-"*50)
print("3️⃣ DROP REDUNDANT COLUMNS")
print("-"*50)

cols_to_drop = ['Product Type', 'Payment Method', 'Shipping Type', 
                'SKU', 'Add-ons Purchased', 'Purchase Date', 'Customer ID']

print(f"Dropping: {cols_to_drop}")
df = df.drop(columns=cols_to_drop)

print(f"\n✅ Dropped {len(cols_to_drop)} columns")
print(f"\n📊 Final columns: {len(df.columns)}")
print(f"\n📋 Column list:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col}")


--------------------------------------------------
3️⃣ DROP REDUNDANT COLUMNS
--------------------------------------------------
Dropping: ['Product Type', 'Payment Method', 'Shipping Type', 'SKU', 'Add-ons Purchased', 'Purchase Date', 'Customer ID']

✅ Dropped 7 columns

📊 Final columns: 30

📋 Column list:
    1. Age
    2. Gender
    3. Loyalty Member
    4. Rating
    5. Order Status
    6. Total Price
    7. Unit Price
    8. Quantity
    9. Add-on Total
   10. Purchase_Month
   11. Purchase_DayOfWeek
   12. Is_Weekend
   13. Has_Addon
   14. Addon_Count
   15. Is_Repeat_Customer
   16. Product_Headphones
   17. Product_Laptop
   18. Product_Smartphone
   19. Product_Smartwatch
   20. Product_Tablet
   21. Payment_Bank Transfer
   22. Payment_Cash
   23. Payment_Credit Card
   24. Payment_Debit Card
   25. Payment_PayPal
   26. Shipping_Expedited
   27. Shipping_Express
   28. Shipping_Overnight
   29. Shipping_Same Day
   30. Shipping_Standard


---
## 🔧 Step 3.5: Handle Outliers

**Strategy**: Cap at 1st and 99th percentile (Winsorization)

**Columns to cap**:
- Total Price (1.92% outliers)
- Add-on Total (1.24% outliers)

In [11]:
# Step 3.5: Handle Outliers (Capping)
print("="*60)
print("STEP 3.5: HANDLE OUTLIERS (CAPPING)")
print("="*60)

cols_to_cap = ['Total Price', 'Add-on Total']

print("\n📊 BEFORE CAPPING:")
print("-"*50)

for col in cols_to_cap:
    p1 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    outliers_high = (df[col] > p99).sum()
    
    print(f"\n{col}:")
    print(f"   Min: {df[col].min():.2f} | Max: {df[col].max():.2f}")
    print(f"   1st percentile: {p1:.2f}")
    print(f"   99th percentile: {p99:.2f}")
    print(f"   Outliers above 99%: {outliers_high}")

STEP 3.5: HANDLE OUTLIERS (CAPPING)

📊 BEFORE CAPPING:
--------------------------------------------------

Total Price:
   Min: 20.75 | Max: 11396.80
   1st percentile: 20.75
   99th percentile: 10268.52
   Outliers above 99%: 200

Add-on Total:
   Min: 0.00 | Max: 292.77
   1st percentile: 0.00
   99th percentile: 228.87
   Outliers above 99%: 200


In [12]:
# Apply capping
print("\n" + "="*60)
print("APPLYING CAPPING (1st - 99th percentile)")
print("="*60)

for col in cols_to_cap:
    p1 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    
    before_min = df[col].min()
    before_max = df[col].max()
    
    df[col] = df[col].clip(lower=p1, upper=p99)
    
    after_min = df[col].min()
    after_max = df[col].max()
    
    print(f"\n✅ {col}:")
    print(f"   Before: [{before_min:.2f}, {before_max:.2f}]")
    print(f"   After:  [{after_min:.2f}, {after_max:.2f}]")


APPLYING CAPPING (1st - 99th percentile)

✅ Total Price:
   Before: [20.75, 11396.80]
   After:  [20.75, 10268.52]

✅ Add-on Total:
   Before: [0.00, 292.77]
   After:  [0.00, 228.87]


---
## 🔧 Step 3.6: Feature Scaling

**Method**: StandardScaler (Mean=0, Std=1)

**Columns to scale**: Continuous numerical features

**Columns NOT scaled**: Binary and One-Hot encoded features

In [13]:
# Step 3.6: Feature Scaling
print("="*60)
print("STEP 3.6: FEATURE SCALING (StandardScaler)")
print("="*60)

# Identify columns to scale (continuous numerical)
cols_to_scale = ['Age', 'Rating', 'Total Price', 'Unit Price', 'Quantity', 
                 'Add-on Total', 'Purchase_Month', 'Purchase_DayOfWeek', 'Addon_Count']

# Columns NOT to scale (binary/one-hot)
cols_no_scale = [col for col in df.columns if col not in cols_to_scale]

print(f"\n📊 Columns to SCALE ({len(cols_to_scale)}):")
for col in cols_to_scale:
    print(f"   • {col}: range [{df[col].min():.2f}, {df[col].max():.2f}]")

print(f"\n📊 Columns NOT scaled ({len(cols_no_scale)}): {cols_no_scale}")

STEP 3.6: FEATURE SCALING (StandardScaler)

📊 Columns to SCALE (9):
   • Age: range [18.00, 80.00]
   • Rating: range [1.00, 5.00]
   • Total Price: range [20.75, 10268.52]
   • Unit Price: range [20.75, 1139.68]
   • Quantity: range [1.00, 10.00]
   • Add-on Total: range [0.00, 228.87]
   • Purchase_Month: range [1.00, 12.00]
   • Purchase_DayOfWeek: range [0.00, 6.00]
   • Addon_Count: range [0.00, 3.00]

📊 Columns NOT scaled (21): ['Gender', 'Loyalty Member', 'Order Status', 'Is_Weekend', 'Has_Addon', 'Is_Repeat_Customer', 'Product_Headphones', 'Product_Laptop', 'Product_Smartphone', 'Product_Smartwatch', 'Product_Tablet', 'Payment_Bank Transfer', 'Payment_Cash', 'Payment_Credit Card', 'Payment_Debit Card', 'Payment_PayPal', 'Shipping_Expedited', 'Shipping_Express', 'Shipping_Overnight', 'Shipping_Same Day', 'Shipping_Standard']


In [14]:
# Apply StandardScaler
print("\n📊 BEFORE SCALING:")
print(df[cols_to_scale].describe().round(2))

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print("\n📊 AFTER SCALING:")
print(df[cols_to_scale].describe().round(2))

# Verification
print("\n✅ All scaled columns now have Mean ≈ 0 and Std ≈ 1")

# Save scaler parameters for future use
scaler_params = pd.DataFrame({
    'feature': cols_to_scale,
    'mean': scaler.mean_,
    'std': scaler.scale_
})
print("\n📋 Scaler Parameters:")
print(scaler_params)


📊 BEFORE SCALING:
            Age    Rating  Total Price  Unit Price  Quantity  Add-on Total  \
count  20000.00  20000.00     20000.00    20000.00  20000.00      20000.00   
mean      48.99      3.09      3168.85      578.63      5.49         62.06   
std       18.04      1.22      2510.80      312.27      2.87         57.48   
min       18.00      1.00        20.75       20.75      1.00          0.00   
25%       33.00      2.00      1139.68      361.18      3.00          7.62   
50%       49.00      3.00      2534.49      463.96      5.00         51.70   
75%       65.00      4.00      4639.60      791.19      8.00         93.84   
max       80.00      5.00     10268.52     1139.68     10.00        228.87   

       Purchase_Month  Purchase_DayOfWeek  Addon_Count  
count        20000.00             20000.0     20000.00  
mean             5.71                 3.0         1.51  
std              3.12                 2.0         1.11  
min              1.00                 0.0         

---
## 🔧 Step 3.7: Handle Class Imbalance

**Target Distribution**:
- Completed (1): 67.2%
- Cancelled (0): 32.8%

**Decision**: 67:33 ratio is mild — use `class_weight='balanced'` during model training instead of SMOTE

**Rationale**: Avoids creating synthetic data, preserves original distribution

In [15]:
# Step 3.7: Class Imbalance Decision
print("="*60)
print("STEP 3.7: CLASS IMBALANCE DECISION")
print("="*60)

print("\n📊 Target Variable Distribution:")
print(df['Order Status'].value_counts())
print(f"\n   • Completed (1): {(df['Order Status'] == 1).mean()*100:.1f}%")
print(f"   • Cancelled (0): {(df['Order Status'] == 0).mean()*100:.1f}%")

print("\n📋 DECISION:")
print("   • Skip SMOTE (67:33 is mild imbalance)")
print("   • Use class_weight='balanced' during model training")
print("   • Rationale: Preserves original data, avoids synthetic samples")

STEP 3.7: CLASS IMBALANCE DECISION

📊 Target Variable Distribution:
Order Status
1    13432
0     6568
Name: count, dtype: int64

   • Completed (1): 67.2%
   • Cancelled (0): 32.8%

📋 DECISION:
   • Skip SMOTE (67:33 is mild imbalance)
   • Use class_weight='balanced' during model training
   • Rationale: Preserves original data, avoids synthetic samples


---
## 🔧 Step 3.8: Train-Test Split

**Parameters**:
- Test size: 20% (4,000 records)
- Stratify: Yes (maintain class ratio)
- Random state: 42 (reproducibility)

In [16]:
# Step 3.8: Train-Test Split
print("="*60)
print("STEP 3.8: TRAIN-TEST SPLIT")
print("="*60)

# Separate features (X) and target (y)
target_col = 'Order Status'
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"\n📊 Dataset shape:")
print(f"   • Features (X): {X.shape}")
print(f"   • Target (y): {y.shape}")
print(f"   • Feature columns: {X.shape[1]}")

# Perform stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print("\n" + "="*60)
print("SPLIT RESULTS")
print("="*60)

print(f"\n✅ Training set:")
print(f"   • X_train: {X_train.shape}")
print(f"   • y_train: {y_train.shape}")
print(f"   • Completed (1): {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)")
print(f"   • Cancelled (0): {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")

print(f"\n✅ Test set:")
print(f"   • X_test: {X_test.shape}")
print(f"   • y_test: {y_test.shape}")
print(f"   • Completed (1): {(y_test == 1).sum()} ({(y_test == 1).mean()*100:.1f}%)")
print(f"   • Cancelled (0): {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)")

print("\n✅ Stratification verified: ratios match in train and test")

STEP 3.8: TRAIN-TEST SPLIT

📊 Dataset shape:
   • Features (X): (20000, 29)
   • Target (y): (20000,)
   • Feature columns: 29

SPLIT RESULTS

✅ Training set:
   • X_train: (16000, 29)
   • y_train: (16000,)
   • Completed (1): 10746 (67.2%)
   • Cancelled (0): 5254 (32.8%)

✅ Test set:
   • X_test: (4000, 29)
   • y_test: (4000,)
   • Completed (1): 2686 (67.2%)
   • Cancelled (0): 1314 (32.9%)

✅ Stratification verified: ratios match in train and test


In [18]:
# Save processed datasets
print("\n" + "="*60)
print("SAVING PROCESSED DATA")
print("="*60)

# 1. Save FULL preprocessed dataset (before split)
df.to_csv('../data/processed/preprocessed_dataset.csv', index=False)
print("\n💾 Saved: preprocessed_dataset.csv (20,000 records - full cleaned data)")

# 2. Separate features (X) and target (y)
target_col = 'Order Status'
X = df.drop(columns=[target_col])
y = df[target_col]

# 3. Perform stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# 4. Save train and test sets
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('../data/processed/data_train.csv', index=False)
test_df.to_csv('../data/processed/data_test.csv', index=False)

# 5. Save feature list
feature_list = X.columns.tolist()
pd.DataFrame({'feature': feature_list}).to_csv('../data/processed/feature_list.csv', index=False)

# 6. Save scaler parameters
scaler_params.to_csv('../data/processed/scaler_params.csv', index=False)

print("💾 Saved: data_train.csv (16,000 records)")
print("💾 Saved: data_test.csv (4,000 records)")
print("💾 Saved: feature_list.csv (29 features)")
print("💾 Saved: scaler_params.csv (scaling parameters)")


SAVING PROCESSED DATA

💾 Saved: preprocessed_dataset.csv (20,000 records - full cleaned data)
💾 Saved: data_train.csv (16,000 records)
💾 Saved: data_test.csv (4,000 records)
💾 Saved: feature_list.csv (29 features)
💾 Saved: scaler_params.csv (scaling parameters)


---
## ✅ Phase 3 Complete: Summary

| Step | Task | Status |
|------|------|--------|
| 3.1 | Handle missing values | ✅ Gender: mode, Add-ons: "None" |
| 3.2 | Fix data inconsistencies | ✅ PayPal standardized |
| 3.3 | Feature engineering | ✅ 6 new features created |
| 3.4 | Encode categorical variables | ✅ Binary + One-Hot |
| 3.5 | Handle outliers | ✅ Capped at 1st-99th percentile |
| 3.6 | Feature scaling | ✅ StandardScaler applied |
| 3.7 | Class imbalance | ✅ Use class weights (no SMOTE) |
| 3.8 | Train-test split | ✅ 80/20 stratified |

**Final Dataset**:
- Training: 16,000 records × 29 features
- Test: 4,000 records × 29 features
- Target: Order Status (1=Completed, 0=Cancelled)

**Ready for Phase 4: Feature Selection**